In [1]:
import os
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import anndata
import seaborn as sns
import collections
from collections import Counter

## Differentailly expressed genes: DCIS vs IDC

In [ ]:
sample_list_sm_with_idc_dcis = [
    "P1","P3","P4","P5","P6","P7","P8","P9","P10","P11_S1","P12","P14","P15","P16","P26","P27","P30","P32","P33","P34","P35","P36","P37","P38","P39",
    "P40_1","P42","P43_2","P44","P45","P47"
]

In [5]:
noise_metabolites = pd.read_csv('E:/Projects/BRCA/results/SM/filter/noise_metabolites_filtered.csv', index_col=0)

In [6]:
noise_metabolites['gene'] = noise_metabolites['gene'].astype(str)

In [ ]:
for spID in sample_list_sm_with_idc_dcis:
    print(spID)
    adata = sc.read_h5ad(f'E:/Projects/BRCA/Intermediate_Data/SM5u/matched_object/{spID}_match_st.h5ad')
    adata = adata[:,~adata.var_names.isin(noise_metabolites['gene'].tolist())]
    print(adata.shape)
    spID_st = spID.replace('SM','ST')
    region_info = pd.read_csv(f'E:/Projects/BRCA/results/ST/cluster_ljy/all_sample_region/{spID_st}_region.csv', index_col=0)
    adata.obs['region'] = region_info['region'].reindex(adata.obs.index)
    adata.obs['region'] = adata.obs['region'].fillna('undefined')
    
    adata = adata[adata.obs['region'].isin(['DCIS','IDC'])]
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.tl.rank_genes_groups(adata, groupby="region", groups=["IDC"], reference="DCIS", method = "wilcoxon")
    dem = sc.get.rank_genes_groups_df(adata, group=None)
    dem.to_csv(f'E:/Projects/BRCA/results/SM/dem/IDC_vs_DCIS/{spID}_dem_idc_vs_dcis.csv')